# Get price history (500 candles)

For a future analysis

In [43]:
import requests
import pandas as pd
import logging
import time
import datetime
import os
import csv
import asyncio

# from dotenv import load_dotenv  # we cannot use it cause it's not in Anaconda Base list 

# Change path to import from directory above
import sys
sys.path.append('../')

from trading_data_classes import GetDataTradingView, DataWorks
import edgar_client

tv = GetDataTradingView()
dw = DataWorks()

In [37]:
def make_a_record_from_tv(symbol, exchange, interval, n_bars, file_path):
    df = tv.get_hist(           
        symbol = symbol,        #  Instrument name, format like "BTCUSDT"
        exchange = exchange,    #  Exchange, source of the quotes (from which TradingView get quotes)
                                #               format "BINANCE"
        interval = interval,    #  str value like "5" --> means 5 minutes
        n_bars = n_bars,        #  How many bars (candles) we're requesting: 
                                #               1 --> only the last one, up to 10_000 --> for history (paywall after ~10k)
    )
    # file_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), file_path)
    try: 
        timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
        write_header = False
        if not os.path.exists(file_path) or os.path.getsize(file_path) == 0:
            write_header = True
        with open(file_path, 'a', encoding='utf-8') as f:
            if write_header:
                f.write("instrument,timestamp_utc,open_price,high_price,low_price,close_price,record_timestamp_utc\n")
            # Write all rows from the DataFrame
            for idx, row in df.iterrows():
                """
                Writing the row with the following data: 
                
                (0) exchange:instument (symbol)
                (1) candle datettime (pandas to_datetime) with the format 2025-09-13 17:30:00115905.88
                (2) open_price
                (3) high_price
                (4) low_price
                (5) close_price
                (6) record_timestamp (when the record has been put into the file)
                
                """
                f.write(f'{row.iloc[0]},'
                        f'{pd.to_datetime(idx)},'
                        f'{row.iloc[1]},{row.iloc[2]},{row.iloc[3]},{row.iloc[4]},'
                        f'{timestamp}\n')
        dw.write_log_line(text = f"Candle of '{row.iloc[0]}' has written with the time {pd.to_datetime(idx)}")
    except Exception as e:
        logging.error(f"Error writing to log file: {e}")
        print(f"Error writing to log file: {e}")
    return

In [ ]:
# Get 500 rows of every instrument (5 minutes candles)
make_a_record_from_tv(symbol = "MAG7",                                  
                      exchange = "LSE", 
                      interval = "5", 
                      n_bars = 500, 
                      file_path = '../data/mag7.csv')

make_a_record_from_tv(symbol = "BTCUSDT",                                  
                        exchange = "BINANCE", 
                        interval = "5", 
                        n_bars = 500, 
                        file_path = '../data/btcusdt.csv'), 

make_a_record_from_tv(symbol = "TONUSDT",                                 
                        exchange = "BINANCE", 
                        interval = "5", 
                        n_bars = 500, 
                        file_path = '../data/tonusdt.csv'),

Candle of 'LSE:MAG7' has written with the time 2025-09-30 15:35:00+00:00
Candle of 'BINANCE:BTCUSDT' has written with the time 2025-09-30 19:10:00+00:00
Candle of 'BINANCE:TONUSDT' has written with the time 2025-09-30 19:10:00+00:00


(None,)

# Get news history


In [39]:
# Alternative approach: Multiple requests with different strategies to collect diverse news articles
output_csv = '../data/news.csv'

# Initialize CSV file with header
os.makedirs(os.path.dirname(output_csv) or '.', exist_ok=True)
with open(output_csv, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(["instrument", "utc_timestamp", "source", "raw_text"])

total_articles_processed = 0
collected_article_ids = set()  # Track unique articles to avoid duplicates

# Strategy 1: Get latest crypto articles
print("Strategy 1: Fetching latest crypto articles...")
try:
    resp = requests.get(
        "https://data-api.coindesk.com/news/v1/article/list",
        params={
            "lang": "EN", 
            "limit": 200,  # Get more articles in one request
            "source_ids": "coindesk",
            "categories": "News,Stablecoins,Cryptocurrency,BTC,ETH,SOL,LTC,USDC"
        },
        headers={"Content-type": "application/json; charset=UTF-8"},
        timeout=30,
    )
    resp.raise_for_status()
    js = resp.json()
    
    count = 0
    with open(output_csv, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        
        for article in js.get("Data", []):
            article_id = article.get("GUID", f"{article.get('TITLE', '')}_{article.get('PUBLISHED_ON', '')}")
            if article_id in collected_article_ids:
                continue
                
            cats = [c.get("CATEGORY") for c in article.get("CATEGORY_DATA", [])]
            relevant_instruments = []
            
            if "BTC" in cats:
                relevant_instruments.append("BTC")
            if "ETH" in cats:
                relevant_instruments.append("ETH")
            if "SOL" in cats:
                relevant_instruments.append("SOL")
            if "LTC" in cats:
                relevant_instruments.append("LTC")
            if "USDC" in cats:
                relevant_instruments.append("USDC")
                
            if not relevant_instruments:
                if any(cat in cats for cat in ["CRYPTOCURRENCY", "STABLECOINS", "TRADING", "MARKET"]):
                    relevant_instruments.append("CRYPTO_GENERAL")
            
            if relevant_instruments:
                collected_article_ids.add(article_id)
                for instrument in relevant_instruments:
                    utc_ts = datetime.datetime.fromtimestamp(article["PUBLISHED_ON"], tz=datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                    source = article.get("SOURCE_DATA", {}).get("NAME", "Unknown")
                    raw_text = article.get("BODY", "")
                    writer.writerow([instrument, utc_ts, source, raw_text])
                    count += 1
    
    total_articles_processed += count
    print(f"Strategy 1: Collected {len([a for a in collected_article_ids])} unique articles, {count} entries")
    
except Exception as e:
    print(f"Strategy 1 failed: {e}")

time.sleep(3)

# Strategy 2: Try different source combinations (if available)
print("\nStrategy 2: Fetching from broader news sources...")
try:
    resp = requests.get(
        "https://data-api.coindesk.com/news/v1/article/list",
        params={
            "lang": "EN", 
            "limit": 100,
            # Try without source filter to get more sources
        },
        headers={"Content-type": "application/json; charset=UTF-8"},
        timeout=30,
    )
    resp.raise_for_status()
    js = resp.json()
    
    count = 0
    with open(output_csv, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        
        for article in js.get("Data", []):
            article_id = article.get("GUID", f"{article.get('TITLE', '')}_{article.get('PUBLISHED_ON', '')}")
            if article_id in collected_article_ids:
                continue
                
            cats = [c.get("CATEGORY") for c in article.get("CATEGORY_DATA", [])]
            article_text = f"{article.get('TITLE', '')} {article.get('BODY', '')}".upper()
            
            relevant_instruments = []
            
            # Check categories
            if "BTC" in cats:
                relevant_instruments.append("BTC")
            if "ETH" in cats:
                relevant_instruments.append("ETH")
            if "SOL" in cats:
                relevant_instruments.append("SOL")
            if "LTC" in cats:
                relevant_instruments.append("LTC")
            if "USDC" in cats:
                relevant_instruments.append("USDC")
                
            # Also check content for crypto mentions
            if not relevant_instruments:
                if any(crypto in article_text for crypto in ['BITCOIN', 'BTC']):
                    relevant_instruments.append("BTC")
                elif any(crypto in article_text for crypto in ['ETHEREUM', 'ETH']):
                    relevant_instruments.append("ETH")
                elif any(crypto in article_text for crypto in ['SOLANA', 'SOL']):
                    relevant_instruments.append("SOL")
                elif any(crypto in article_text for crypto in ['LITECOIN', 'LTC']):
                    relevant_instruments.append("LTC")
                elif any(crypto in article_text for crypto in ['USDC', 'USD COIN']):
                    relevant_instruments.append("USDC")
                elif any(keyword in article_text for keyword in ['CRYPTO', 'BLOCKCHAIN', 'DEFI', 'STABLECOIN']):
                    relevant_instruments.append("CRYPTO_GENERAL")
            
            if relevant_instruments:
                collected_article_ids.add(article_id)
                for instrument in relevant_instruments:
                    utc_ts = datetime.datetime.fromtimestamp(article["PUBLISHED_ON"], tz=datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                    source = article.get("SOURCE_DATA", {}).get("NAME", "Unknown")
                    raw_text = article.get("BODY", "")
                    writer.writerow([instrument, utc_ts, source, raw_text])
                    count += 1
    
    total_articles_processed += count
    print(f"Strategy 2: Added {count} new entries")
    
except Exception as e:
    print(f"Strategy 2 failed: {e}")

time.sleep(3)

# Strategy 3: Multiple smaller requests with delays to potentially get different batches
print("\nStrategy 3: Multiple smaller batches...")
for batch in range(3):
    try:
        time.sleep(2)  # Wait between batches
        resp = requests.get(
            "https://data-api.coindesk.com/news/v1/article/list",
            params={
                "lang": "EN", 
                "limit": 100,
                "source_ids": "coindesk"
            },
            headers={"Content-type": "application/json; charset=UTF-8"},
            timeout=30,
        )
        resp.raise_for_status()
        js = resp.json()
        
        batch_count = 0
        with open(output_csv, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, quoting=csv.QUOTE_ALL)
            
            for article in js.get("Data", []):
                article_id = article.get("GUID", f"{article.get('TITLE', '')}_{article.get('PUBLISHED_ON', '')}")
                if article_id in collected_article_ids:
                    continue
                    
                cats = [c.get("CATEGORY") for c in article.get("CATEGORY_DATA", [])]
                article_text = f"{article.get('TITLE', '')} {article.get('BODY', '')}".upper()
                
                relevant_instruments = []
                
                if "BTC" in cats or 'BITCOIN' in article_text or 'BTC' in article_text:
                    relevant_instruments.append("BTC")
                if "ETH" in cats or 'ETHEREUM' in article_text or 'ETH' in article_text:
                    relevant_instruments.append("ETH")
                if "SOL" in cats or 'SOLANA' in article_text or 'SOL' in article_text:
                    relevant_instruments.append("SOL")
                if "LTC" in cats or 'LITECOIN' in article_text or 'LTC' in article_text:
                    relevant_instruments.append("LTC")
                if "USDC" in cats or 'USDC' in article_text:
                    relevant_instruments.append("USDC")
                    
                if not relevant_instruments and (
                    any(cat in cats for cat in ["CRYPTOCURRENCY", "STABLECOINS", "TRADING", "MARKET"]) or
                    any(keyword in article_text for keyword in ['CRYPTO', 'BLOCKCHAIN', 'DEFI'])
                ):
                    relevant_instruments.append("CRYPTO_GENERAL")
                
                if relevant_instruments:
                    collected_article_ids.add(article_id)
                    for instrument in relevant_instruments:
                        utc_ts = datetime.datetime.fromtimestamp(article["PUBLISHED_ON"], tz=datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                        source = article.get("SOURCE_DATA", {}).get("NAME", "Unknown")
                        raw_text = article.get("BODY", "")
                        writer.writerow([instrument, utc_ts, source, raw_text])
                        batch_count += 1
        
        total_articles_processed += batch_count
        print(f"Batch {batch + 1}: Added {batch_count} new entries")
        
    except Exception as e:
        print(f"Batch {batch + 1} failed: {e}")

print(f"\nCompleted all strategies!")
print(f"Total unique articles collected: {len(collected_article_ids)}")
print(f"Total article entries processed: {total_articles_processed}")
print(f"News written to: {os.path.abspath(output_csv)}")

Strategy 1: Fetching latest crypto articles...
Strategy 1 failed: 400 Client Error: Bad Request for url: https://data-api.coindesk.com/news/v1/article/list?lang=EN&limit=200&source_ids=coindesk&categories=News%2CStablecoins%2CCryptocurrency%2CBTC%2CETH%2CSOL%2CLTC%2CUSDC

Strategy 2: Fetching from broader news sources...
Strategy 2: Added 92 new entries

Strategy 3: Multiple smaller batches...
Batch 1: Added 167 new entries
Batch 2: Added 0 new entries
Batch 3: Added 0 new entries

Completed all strategies!
Total unique articles collected: 178
Total article entries processed: 259
News written to: c:\Users\HAL9003\Desktop\Realtime-Trading-Monitoring\data\news.csv


In [40]:
# Debug: Test different API parameters to understand pagination
print("Testing API pagination methods...")

# Test 1: Check if page parameter works
try:
    print("Testing 'page' parameter...")
    resp = requests.get(
        "https://data-api.coindesk.com/news/v1/article/list",
        params={"lang": "EN", "limit": 20, "page": 2, "source_ids": "coindesk"},
        timeout=30
    )
    js = resp.json()
    print(f"Page 2 test: Got {len(js.get('Data', []))} articles")
    if len(js.get('Data', [])) > 0:
        first_article_time = js['Data'][0].get('PUBLISHED_ON', 0)
        print(f"First article timestamp: {first_article_time}")
except Exception as e:
    print(f"Page parameter failed: {e}")

# Test 2: Check response structure for pagination hints
try:
    print("\nTesting response structure...")
    resp = requests.get(
        "https://data-api.coindesk.com/news/v1/article/list",
        params={"lang": "EN", "limit": 5, "source_ids": "coindesk"},
        timeout=30
    )
    js = resp.json()
    print("Response keys:", list(js.keys()))
    if 'Meta' in js:
        print("Meta information:", js['Meta'])
    print(f"Total articles in response: {len(js.get('Data', []))}")
except Exception as e:
    print(f"Structure test failed: {e}")

# Test 3: Try smaller batches and see if we can get more by calling multiple times with slight delays
print("\nTesting multiple small batches...")
unique_titles = set()
for i in range(3):
    try:
        time.sleep(1)  # Small delay between requests
        resp = requests.get(
            "https://data-api.coindesk.com/news/v1/article/list",
            params={"lang": "EN", "limit": 50, "source_ids": "coindesk"},
            timeout=30
        )
        js = resp.json()
        batch_titles = {article.get('TITLE', '') for article in js.get('Data', [])}
        new_titles = batch_titles - unique_titles
        unique_titles.update(batch_titles)
        print(f"Batch {i+1}: {len(batch_titles)} articles, {len(new_titles)} new, {len(unique_titles)} total unique")
    except Exception as e:
        print(f"Batch {i+1} failed: {e}")

Testing API pagination methods...
Testing 'page' parameter...
Page 2 test: Got 20 articles
First article timestamp: 1759248409

Testing response structure...
Response keys: ['Data', 'Err']
Total articles in response: 5

Testing multiple small batches...
Batch 1: 50 articles, 50 new, 50 total unique
Batch 2: 50 articles, 0 new, 50 total unique
Batch 3: 50 articles, 0 new, 50 total unique


In [ ]:
# # Debug: Let's check what categories are actually being returned
# print("Debugging API response to see available categories...")
# resp = requests.get(
#     "https://data-api.coindesk.com/news/v1/article/list",
#     params={"lang": "EN", "limit": 10, "source_ids": "coindesk", "categories": "News,Stablecoins,Cryptocurrency,BTC,TON,MAG7"},
#     headers={"Content-type": "application/json; charset=UTF-8"},
#     timeout=30,
# )

# resp.raise_for_status()
# js = resp.json()

# print(f"Total articles returned: {len(js.get('Data', []))}")
# all_categories = set()

# for i, article in enumerate(js.get("Data", [])[:5]):  # Check first 5 articles
#     cats = [c.get("CATEGORY") for c in article.get("CATEGORY_DATA", [])]
#     all_categories.update(cats)
#     print(f"\nArticle {i+1} categories: {cats}")
#     print(f"Article {i+1} title: {article.get('TITLE', 'No title')[:100]}...")

# print(f"\nAll unique categories found: {sorted(all_categories)}")

In [41]:
# Fetch combined SEC filings (inline) and write to data/sec_filings_combined.csv
out = edgar_client.get_and_write_combined_btc_ton_mag7(output_dir='data', 
                                                       btc_limit=150, 
                                                       ton_limit=150, 
                                                       mag7_limit_each=150)
print(f"SEC combined written to: {out}")

SEC combined written to: c:\Users\HAL9003\Desktop\Realtime-Trading-Monitoring\data/sec_filings_combined.csv


##  Drop duplicates after recording historical data

In [44]:
def remove_duplicates_from_csv(file_path):
    """
    Read a CSV file, remove duplicate rows, and save back to the same file.
    
    Args:
        file_path (str): Path to the CSV file to process
        
    Returns:
        tuple: (original_count, final_count, duplicates_removed)
    """
    try:
        # Read the CSV file
        df = pd.read_csv(file_path)
        original_count = len(df)
        
        # Remove duplicates (keeps first occurrence)
        df_clean = df.drop_duplicates()
        final_count = len(df_clean)
        duplicates_removed = original_count - final_count
        
        # Save back to the same file
        df_clean.to_csv(file_path, index=False)
        
        print(f"File: {file_path}")
        print(f"Original rows: {original_count}")
        print(f"Final rows: {final_count}")
        print(f"Duplicates removed: {duplicates_removed}")
        
        return original_count, final_count, duplicates_removed
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None, None, None

In [45]:
# Example usage: Remove duplicates from all data files
data_files = [
    '../data/btcusdt.csv',
    '../data/tonusdt.csv', 
    '../data/mag7.csv',
    '../data/news.csv',
    '../data/sec_filings_combined.csv'
]

for file_path in data_files:
    if os.path.exists(file_path):
        remove_duplicates_from_csv(file_path)
        print("-" * 50)
    else:
        print(f"File not found: {file_path}")
        print("-" * 50)

File: ../data/btcusdt.csv
Original rows: 35
Final rows: 35
Duplicates removed: 0
--------------------------------------------------
File: ../data/tonusdt.csv
Original rows: 59
Final rows: 59
Duplicates removed: 0
--------------------------------------------------
File: ../data/mag7.csv
Original rows: 67
Final rows: 67
Duplicates removed: 0
--------------------------------------------------
File: ../data/news.csv
Original rows: 259
Final rows: 259
Duplicates removed: 0
--------------------------------------------------
File: ../data/sec_filings_combined.csv
Original rows: 1292
Final rows: 1292
Duplicates removed: 0
--------------------------------------------------


In [ ]:
# TODO: add history collection for Fed rates (requires fed_rates_scraper.py to be rewritten) 